## Code by Raventh

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\DELL\\OneDrive\\Desktop\\Let us build\\AI-book-recommender\\research'

In [3]:
import os

os.chdir(r"C:\Users\DELL\OneDrive\Desktop\Let us build\AI-book-recommender")

In [4]:
%pwd

'C:\\Users\\DELL\\OneDrive\\Desktop\\Let us build\\AI-book-recommender'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    vectorizer_path: Path
    model_path: Path
    n_neighbors: int
    metric: str
    algorithm: str

In [6]:
from Book_recommender.constant import *
from Book_recommender.utils.common import read_yaml,create_directories
from Book_recommender.config.configuration import configurationManager

In [7]:
config = configurationManager()
model_trainer_config = config.get_model_trainer_config()

[2026-06-01 22:46:18,985: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-06-01 22:46:19,025: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-06-01 22:46:19,035: INFO: common: created directory at artifacts]
[2026-06-01 22:46:19,042: INFO: common: created directory at artifacts/model_trainer]


In [8]:
class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            vectorizer_path = config.vectorizer_path,
            model_path = config.model_path,
            n_neighbors = config.n_neighbors,
            metric = config.metric,
            algorithm = config.algorithm
        )
        return model_trainer_config

In [9]:
from Book_recommender.logging.log import logger
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = joblib.load(
    model_trainer_config.vectorizer_path
)

In [10]:
data = pd.read_csv(model_trainer_config.data_path)
data.head()

,title,authors,average_rating,isbn13,language_code,num_pages,ratings_count,description,genre_clean,combined_features
0,The New Testament,Richmond Lattimore,4.32,9780865475243,eng,608,109,No Description,Bible,the new testament richmond lattimore bible no ...
1,The Art of Loving by Erich Fromm: A True Story...,Lala Okamoto,1.67,9784990327507,eng,227,3,A True Story of a Japanese Woman,NaN,the art of loving by erich fromm a true story ...
2,Mere Christianity: Abolition of Man (Bonus Fea...,C.S. Lewis/Geoffrey Howard/Robert Whitefield/R...,4.32,9780786174362,eng,6,43,No Description,Apologetics,mere christianity abolition of man bonus featu...
3,The Four Loves,C.S. Lewis,4.14,9780006280897,en-US,170,34681,No Description,Love (Theology),the four loves cs lewis love theology no descr...
4,The Play Soldier,Chet Green,4.50,9781591136446,eng,328,2,No Description,NaN,the play soldier chet green no description


In [11]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
class ModelTrainer:
    def __init__(self, config):
        self.config = config
        self.model = None

    def initialize_model(self):
        self.model = NearestNeighbors(
        n_neighbors=self.config.n_neighbors,
        metric=self.config.metric,
        algorithm=self.config.algorithm
    )

    def fit_model(self, tfidf_matrix):
        self.model.fit(tfidf_matrix)

    def save(self):
        joblib.dump(
        self.model,
        self.config.model_path
    )

: 

In [13]:
trainer = ModelTrainer(model_trainer_config)

In [14]:
trainer.initialize_model()

In [16]:
tfidf_matrix = vectorizer.transform(
    data["combined_features"]
)

In [17]:
trainer.fit_model(tfidf_matrix)

In [18]:
trainer.save()